In [1]:
import numpy as np
import pandas as pd
import h5py
from sklearn.decomposition import IncrementalPCA
import joblib
import os

In [2]:
results_dir = "Wang2025Nature"

n_components = 50
chunk_size = 10_000

uce_file = "uce.h5"
#group_name = "Brain_NervousSystem"
group_name = ""
ipca_outfile = f"ipca_model_{group_name}.pkl" if group_name else "ipca_model.pkl"
pca_outfile = f"pca_{group_name}.h5" if group_name else "pca.h5"

In [3]:
# handle to uce.h5 
fuce = h5py.File(os.path.join(results_dir, uce_file), "r")
uce = fuce["data"] # h5 handle
print(uce.shape)

(232328, 1280)


## IncrementalPCA - Fit
assuming all data already preshuffled

In [4]:
# Fit PCA in streaming mode
ipca = IncrementalPCA(n_components=n_components)

iteration =0
total_len = uce.shape[0]
#chunks_iter = int(batch_size/chunk_size)
for i in range(0, total_len, chunk_size):
    iteration = iteration + 1
    print ("processing: cell", i, iteration)
    batch = uce[i:i+chunk_size, :]
    ipca.partial_fit(batch)

processing: cell 0 1
processing: cell 10000 2
processing: cell 20000 3
processing: cell 30000 4
processing: cell 40000 5
processing: cell 50000 6
processing: cell 60000 7
processing: cell 70000 8
processing: cell 80000 9
processing: cell 90000 10
processing: cell 100000 11
processing: cell 110000 12
processing: cell 120000 13
processing: cell 130000 14
processing: cell 140000 15
processing: cell 150000 16
processing: cell 160000 17
processing: cell 170000 18
processing: cell 180000 19
processing: cell 190000 20
processing: cell 200000 21
processing: cell 210000 22
processing: cell 220000 23
processing: cell 230000 24


In [5]:
# save PCA fit model -- since it takes a long time to run
joblib.dump(ipca, os.path.join(results_dir, ipca_outfile))

['Wang2025Nature/ipca_model.pkl']

## IncrementalPCA - Transform

In [6]:
# read model back

ipca = joblib.load(os.path.join(results_dir, ipca_outfile))
ipca_outfile


'ipca_model.pkl'

In [7]:
# Apply ipca.transform in batches

# Create output array (e.g., 36M × 50)
#fpca.close()
n_rows = uce.shape[0]
fpca =  h5py.File(os.path.join(results_dir, pca_outfile), "w")

X_pca = fpca.create_dataset(
    "pca",
    shape=(n_rows, ipca.n_components),
    dtype="float32",
    chunks=(chunk_size, ipca.n_components),
    compression="gzip"
)

# incremental transform
for i in range(0, n_rows, chunk_size):
    batch = uce[i:i + chunk_size,:]
    transformed = ipca.transform(batch)
    X_pca[i:i + len(transformed)] = transformed
    print(f"Transformed rows {i} to {i + len(transformed)}")
print("PCA output successfully saved to disk.")
print(X_pca.shape)

Transformed rows 0 to 10000
Transformed rows 10000 to 20000
Transformed rows 20000 to 30000
Transformed rows 30000 to 40000
Transformed rows 40000 to 50000
Transformed rows 50000 to 60000
Transformed rows 60000 to 70000
Transformed rows 70000 to 80000
Transformed rows 80000 to 90000
Transformed rows 90000 to 100000
Transformed rows 100000 to 110000
Transformed rows 110000 to 120000
Transformed rows 120000 to 130000
Transformed rows 130000 to 140000
Transformed rows 140000 to 150000
Transformed rows 150000 to 160000
Transformed rows 160000 to 170000
Transformed rows 170000 to 180000
Transformed rows 180000 to 190000
Transformed rows 190000 to 200000
Transformed rows 200000 to 210000
Transformed rows 210000 to 220000
Transformed rows 220000 to 230000
Transformed rows 230000 to 232328
PCA output successfully saved to disk.
(232328, 50)


In [ ]:
fpca.close()
fuce.close()

In [ ]:
n_neighbors = 20
n_jobs = 8

In [ ]:
# load precomputed Knn
data = np.load(os.path.join(results_dir, "knn_30.npz"))
indices = data["indices"]
distances = data["distances"]
precomputed_knn = (indices, distances)

In [ ]:
# Run UMAP using the precomputed neighbors
import umap
reducer = umap.UMAP(
    n_neighbors=n_neighbors,
    n_components=2,
    precomputed_knn=precomputed_knn,
    metric="precomputed",  # important
    n_jobs = n_jobs,
    # random_state=42,
    verbose=True
)

X_umap = reducer.fit_transform(X_pca)

In [ ]:
X_umap.shape

In [ ]:
# Save UMAP
with h5py.File(os.path.join(results_dir, "umap.h5"), "w") as f:
    f.create_dataset("umap", data=X_umap, compression="gzip")

In [ ]:
# # read umap back
fumap = h5py.File(os.path.join(results_dir, "umap.h5"), "r")
X_umap = fumap["umap"] # h5 handle
print(X_umap.shape)

In [ ]:
import matplotlib.pyplot as plt

# downsample
#indices = np.random.choice(X_umap.shape[0], size=500_000, replace=False)
plt.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, alpha=0.5)
plt.axis('off')
plt.title("UMAP projection")
plt.show()

In [ ]:
# get post uce gene expression data
import anndata
adata = anndata.read_h5ad(os.path.join(results_dir, "c4b03352-af8d-492a-8d6b-40f304e0a122_uce_adata.h5ad"))
adata

In [ ]:
# umap using gene exprssion by myslef
import scanpy as sc

# 1. Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 2. Highly variable genes
sc.pp.highly_variable_genes(adata)
adata = adata[:, adata.var["highly_variable"]]

# 3. Scale (optional but common)
sc.pp.scale(adata, max_value=10)

# 4. PCA
sc.tl.pca(adata, svd_solver="arpack")

# 5. Compute neighbors
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)

# 6. Run UMAP
sc.tl.umap(adata)

# 7. Plot UMAP
sc.pl.umap(adata, color="DRD1")  # or any gene, cluster, metadata
adata

In [ ]:
adata.obsm['X_uce_umap']=X_umap
adata

In [ ]:
adata.write_h5ad(os.path.join(results_dir, "c4b03352-af8d-492a-8d6b-40f304e0a122_uce_adata.h5ad"))

In [ ]:
adata.var

In [ ]:
import scanpy as sc
sc.pl.embedding(adata, "X_umap", color=["DRD1","DRD2","dissection"], gene_symbols="Gene", vmin=0, vmax=3, legend_loc=None)

In [ ]:
import scanpy as sc
sc.pl.embedding(adata, "X_uce_umap", color=["DRD1","DRD2","dissection"], gene_symbols="Gene", vmin=0, vmax=3, legend_loc=None)

get cell type data

In [ ]:
# cell type
cell_type = pd.read_csv(os.path.join(results_dir, "cell_type.tsv.gz"), sep="\t", compression='gzip')

In [ ]:
cell_type.head()

In [ ]:
assert (len(cell_type) == len(X_umap))

In [ ]:
# Extract the cell_type column
labels = cell_type["cell_type"].astype(str).values[indices,]
labels.shape

In [ ]:
# Encode cell type labels to integers for color mapping
encoded_labels, unique_labels = pd.factorize(labels)
len(unique_labels)

In [ ]:
import matplotlib.patches as mpatches

plt.scatter(X_umap[indices, 0], X_umap[indices, 1], 
            c=encoded_labels, cmap="tab20", 
            s=0.1, alpha=0.5)
plt.axis('off')
plt.title("UMAP colored by cell type (500K subset)")

'''
# Create custom legend handles
legend_elements = [
    mpatches.Patch(color=plt.cm.tab20(i / len(unique_labels)), label=label)
    for i, label in enumerate(unique_labels)
]

# Add legend (adjust number of columns or fontsize as needed)
plt.legend(handles=legend_elements, title="Cell Type", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
plt.tight_layout()
'''

plt.show()